# ViTs & CLIP: The Transformer Revolution in Vision

Reach for this when you need: 
- Reference for non-convolutional vision architectures.
- To understand Patching and Positional Embeddings in images.
- Implementing multimodal zero-shot classification with CLIP.

In [ ]:
import torch
import timm
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Vision Transformers (ViT)

Instead of convolutions, ViT splits images into patches (e.g. 16x16) and treats them as sequence tokens.

| Component | Description | Usage |
| :--- | :--- | :--- |
| `Patch Projection` | Conv2d with stride=kernel_size | Linear projection of pixels to tokens |
| `CLS Token` | Extra learned token | Abstract representation of the whole image |
| `Position Embeds` | Added to tokens | Restoration of spatial information |

In [ ]:
# Load ViT from timm (industry standard)
vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0).to(device)
vit.eval()

img = torch.randn(1, 3, 224, 224).to(device)
features = vit(img)
print(f"ViT Features shape: {features.shape}") # [1, 768]

## 2. CLIP (Contrastive Language-Image Pre-training)

Connects text and images in a shared embedding space. Allows for zero-shot classification WITHOUT retraining.

✅ **Use when**: Broad classification tasks where classes aren't known at training time.
❌ **Don't use when**: High-precision, fine-grained classification is required (e.g. medical imaging).

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.new('RGB', (224, 224), color = 'red') # Placeholder image
inputs = processor(text=["a photo of a cat", "a photo of a dog", "red background"], images=image, return_tensors="pt", padding=True).to(device)

outputs = model(**inputs)
probs = outputs.logits_per_image.softmax(dim=1) # Probability for each text label

### Common Pitfalls
- **Data Requirements**: ViTs generally require MUCH more data than CNNs to reach SOTA (though pretrained ones are highly effective).
- **Fixed Resolution**: ViTs are strictly tied to their training resolution (e.g., 224x224); resizing is critical.
- **Tokenizer Match**: For CLIP, always use the specific processor/tokenizer paired with the model weights.

### Key Takeaways
- ViTs lack the 'inductive bias' of convolutions (spatial hierarchy) but scale better with data.
- Multimodal models (CLIP) are the foundation for modern DALLE/Stable Diffusion and RAG-Vision apps.
- Use `timm` for generic ViT backbones and HuggingFace for CLIP-style multimodal models.